# ElephantGrow – Smart Basil Monitoring System

This project combines:
- Inverted Index Search Engine
- RAG (Retrieval Augmented Generation)
- Gemini Vision AI
- Firebase Realtime Database
- Gradio User Interface

The system allows users to:
1. Search academic papers about basil diseases
2. Analyze basil leaf images using AI
3. Simulate IoT sensor readings
4. Store data in Firebase

## Imports + Setup

### Imports and Setup Explanation

This section (`2ZCjknFJDc24`) handles all necessary library imports and initializes key services like NLTK, the Gemini API, and Firebase.

*   **Library Imports**: Essential modules like `gradio`, `pandas`, `random`, `re`, `json`, `hashlib`, `requests`, `datetime`, `collections.defaultdict`, and `PIL.Image` are imported for UI, data manipulation, utility functions, web requests, and image processing.
*   **NLTK Setup**: The Natural Language Toolkit (NLTK) is used for text processing, specifically `wordnet` (for lemmatization), `omw-1.4` (Open Multilingual Wordnet), and `punkt` (for tokenization). A `WordNetLemmatizer` is initialized to reduce words to their base form, which is crucial for the search engine's effectiveness. Error handling ensures the application can still run if NLTK downloads fail.
*   **Gemini API Setup**: This configures access to Google's Gemini AI model. It attempts to retrieve an API key from Colab's user data secrets. If not found, a fallback key is used (though this should ideally be replaced with a secure key for production). The `genai.Client` is then initialized with this key.
*   **Firebase Setup**: This section connects the application to a Firebase Realtime Database. It first tries to get a service account JSON from user data secrets for admin access. If not available, it connects in a "Public Mode" using `UnauthenticatedCreds`. This allows the application to read/write data to the specified Firebase URL (`https://elephantgrow-2026-default-rtdb.firebaseio.com/`). Error handling is included for robust initialization.

In [ ]:
# ==========================================
# 1. IMPORTS
# ==========================================
import gradio as gr
import pandas as pd
import random
import re
import json
import hashlib
import requests
from datetime import datetime
from collections import defaultdict
from PIL import Image

import nltk
from nltk.stem import WordNetLemmatizer

import firebase_admin
from firebase_admin import credentials, db
import google.auth.credentials

from google import genai
from google.colab import userdata

# ==========================================
# 2. NLTK SETUP (MUST RUN BEFORE ARTICLES)
# ==========================================
try:
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    nltk.download('punkt', quiet=True)
    lemmatizer = WordNetLemmatizer()
    print("✅ NLTK & Lemmatizer ready.")
except Exception as e:
    print(f"⚠️ NLTK error: {e}")
    class DummyLemmatizer:
        def lemmatize(self, word): return word
    lemmatizer = DummyLemmatizer()

# ==========================================
# 3. GEMINI API SETUP
# ==========================================
try:

    gemini_api_key = userdata.get("GEMINI_API_KEY")
except Exception:

    gemini_api_key = "**"
    print("⚠️ Secret GEMINI_API_KEY not found. Using fallback key.")

client = genai.Client(api_key=gemini_api_key)

class UnauthenticatedCreds(google.auth.credentials.AnonymousCredentials):
    def refresh(self, request): pass
    @property
    def valid(self): return True

firebase_json = None
try:
    firebase_json = userdata.get("FIREBASE_SERVICE_ACCOUNT_JSON")
except Exception:
    pass

if not firebase_admin._apps:
    try:
        if firebase_json:

            cred = credentials.Certificate(json.loads(firebase_json))
            firebase_admin.initialize_app(cred, {
                "databaseURL": "https://elephantgrow-2026-default-rtdb.firebaseio.com/"
            })
            print("✅ Connected to Firebase as Admin.")
        else:

            firebase_admin.initialize_app(UnauthenticatedCreds(), {
                "databaseURL": "https://elephantgrow-2026-default-rtdb.firebaseio.com/"
            })
            print("✅ Connected to Firebase in Public Mode (No Secret Needed).")
    except Exception as e:
        print(f"❌ Firebase initialization failed: {e}")
else:
    print("ℹ️ Firebase already initialized.")

print("🚀 Setup Complete!")

✅ NLTK & Lemmatizer ready.
⚠️ Secret GEMINI_API_KEY not found. Using fallback key.
✅ Connected to Firebase in Public Mode (No Secret Needed).
🚀 Setup Complete!


### index words

### Index Words Explanation

This cell (`DXCd7KYW8bJH`) retrieves a set of predefined "index words" from the Firebase Realtime Database. These words are critical for the search engine, as only articles containing these specific keywords will be indexed. This approach allows for a focused and controlled search space relevant to basil diseases and growth.

In [ ]:
INDEX_WORDS = set(db.reference("index_words").get())

## Inverted Index and RAG Search Engine

### Inverted Index and RAG Search Engine Explanation

This section (`mqPnL6EJsxe5`) defines the core logic for the RAG (Retrieval Augmented Generation) search engine. It allows users to search academic papers and retrieves relevant content based on their queries.

*   **`STOP_WORDS`**: A set of common English words (like "the", "is", "a") that are typically ignored in search queries because they don't carry significant meaning for retrieval.
*   **`BasilSearchEngine` Class**: This class encapsulates the search engine functionality.
    *   `__init__`: Initializes an empty `inverted_index` (a dictionary where keys are words and values are lists of document IDs containing that word), `titles`, and `urls` lists.
    *   `build_index(titles, contents, urls)`: This method populates the inverted index. It takes lists of article titles, contents (summaries), and URLs. For each article, it cleans and lemmatizes the text, then adds the document ID to the `inverted_index` for any word that is part of the `INDEX_WORDS` set. This ensures only relevant, pre-approved terms are indexed.
    *   `process_query(query)`: Cleans and lemmatizes the user's search query, removing stop words and short words, to prepare it for matching against the index.
    *   `search(query)`: This is the main RAG implementation. It performs both retrieval and a simulated augmented generation:
        *   **Retrieval Phase**: It processes the query, identifies relevant documents based on the `query_words` and the `inverted_index`. It supports `AND`/`OR` logic (though `OR` is currently implemented). It also calculates a basic `scores` for each document based on word frequency in content and title, used for ranking.
        *   **Augmented Generation Phase**: It formats the retrieved documents (title, URL, snippet) and then adds a "simulated AI Insight" to each result. This insight mimics what a large language model might generate, explaining the relevance of the article to the query and offering general advice, demonstrating the "augmented generation" aspect of RAG.
    *   `retrieve_context(query, top_k=3)`: This helper method is similar to `search` but specifically designed to retrieve the most relevant articles (up to `top_k`) as raw text context for the Gemini AI model. It focuses on providing a concise summary of relevant information to guide the AI's analysis.

In [ ]:
# ==========================================
# SEARCH ENGINE WITH RAG
# ==========================================

STOP_WORDS = {
    "the", "is", "at", "which", "on", "and", "a", "an", "to", "in", "of", "for",
    "with", "it", "that", "by", "from", "this", "are", "was", "were", "be", "as",
    "or", "but", "not", "can", "has", "have", "had"
}

class BasilSearchEngine:
    def __init__(self):
        self.inverted_index = defaultdict(list)
        self.titles = []
        self.urls = []

    def build_index(self, titles, contents, urls):
        """Build inverted index using only the 20 selected index words."""

        self.titles = titles
        self.doc_content = dict(enumerate(contents))
        self.urls = urls
        self.inverted_index.clear()

        for doc_id, text in enumerate(contents):
            clean_text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower())

            words = [
                lemmatizer.lemmatize(w)
                for w in clean_text.split()
            ]

            seen = set()

            for word in words:
                if word in INDEX_WORDS and word not in seen:
                    self.inverted_index[word].append(doc_id)
                    seen.add(word)

        # Make sure all 20 index words exist in the index,
        # even if some words do not appear in the article summaries.
        for word in INDEX_WORDS:
            self.inverted_index[word]

        return (
            f"Index built with {len(self.inverted_index)} index words "
            f"from {len(titles)} articles."
        )

    def process_query(self, query):
        """Clean and lemmatize user query."""

        clean_query = re.sub(r'[^a-zA-Z0-9\s]', ' ', query.lower())

        query_words = [
            lemmatizer.lemmatize(w)
            for w in clean_query.split()
            if w not in STOP_WORDS and len(w) > 2
        ]

        return query_words

    def search(self, query):
        """RAG Implementation: Retrieval + simulated augmented generation."""

        if not query or not query.strip():
            return "Please enter a search term."

        query_words = self.process_query(query)

        use_or = " OR " in query.upper()

        results = set()
        scores = defaultdict(int)

        # === RETRIEVAL PHASE ===
        for word in query_words:

            if word in self.inverted_index:
                docs = set(self.inverted_index[word])

                # Ignore index words that exist but have no documents
                if not docs:
                    continue

                if not results:
                    results = docs
                else:
                    if use_or:
                        results = results.union(docs)
                    else:
                        results = results.intersection(docs)

                # Ranking score
                for doc_id in docs:
                    content = self.doc_content[doc_id].lower()
                    title = self.titles[doc_id].lower()

                    scores[doc_id] += content.count(word)
                    scores[doc_id] += title.count(word) * 2

        if not results:
            return "No relevant academic documents found."

        # === AUGMENTED GENERATION PHASE ===
        output = f"RAG Results for: **{query}**\n\n"

        ranked_results = sorted(
            results,
            key=lambda doc_id: scores[doc_id],
            reverse=True
        )

        for doc_id in ranked_results:
            title = self.titles[doc_id]
            url = self.urls[doc_id]
            snippet = self.doc_content.get(doc_id, "")[:480] + "..."

            output += f"**{title}**\n"
            output += f"{url}\n"
            output += f"**Relevant Excerpt:** {snippet}\n\n"

            # Simulated AI generation
            output += (
                f"**🧠 AI Insight:** This paper is relevant to **{query}** "
                "because it discusses basil disease, resistance, environmental "
                "conditions, or growth management. "
            )

            output += (
                "The retrieved research suggests that environmental control, "
                "resistant varieties, and proper management practices can help "
                "reduce disease pressure and improve basil health.\n\n"
            )

            output += "-" * 70 + "\n\n"

        return output

    def retrieve_context(self, query, top_k=3):
        """Retrieve the most relevant articles as context for Gemini."""

        if not query or not query.strip():
            return "No query provided."

        query_words = self.process_query(query)

        scores = defaultdict(int)

        for word in query_words:
            if word in self.inverted_index:
                docs = self.inverted_index[word]

                for doc_id in docs:
                    content = self.doc_content[doc_id].lower()
                    title = self.titles[doc_id].lower()

                    scores[doc_id] += content.count(word)
                    scores[doc_id] += title.count(word) * 2

        if not scores:
            return "No relevant academic context found."

        ranked_docs = sorted(
            scores.keys(),
            key=lambda doc_id: scores[doc_id],
            reverse=True
        )[:top_k]

        context = ""

        for doc_id in ranked_docs:
            context += f"Title: {self.titles[doc_id]}\n"
            context += f"Summary: {self.doc_content[doc_id]}\n"
            context += f"URL: {self.urls[doc_id]}\n\n"

        return context

## Academic Knowledge Base

### Academic Knowledge Base Explanation

This section (`rR6Y_Zv9PtzI` and `pAd2EG9CgyIr`) sets up the knowledge base that the search engine will use. It contains a small dataset of simulated academic articles related to basil diseases.

*   **`titles`, `contents`, `urls`**: These lists store the metadata and content for five academic articles. The `contents` are short summaries or abstracts of the papers.
*   **`academic_context`**: This string concatenates the titles, summaries, and URLs of all articles into a single block of text. While not directly used by the `BasilSearchEngine` for indexing, it serves as a pre-compiled resource, potentially for direct display or for a simpler context retrieval if needed.
*   **`engine = BasilSearchEngine()`**: An instance of the `BasilSearchEngine` is created.
*   **`engine.build_index(titles, contents, urls)`**: The `build_index` method is called to process the academic articles and build the inverted index. This makes the articles searchable by the defined `INDEX_WORDS`.
*   **Inverted Index Display (`pAd2EG9CgyIr`)**: This code block iterates through the `engine.inverted_index` and prints each indexed word along with the document IDs (indices in the `titles`, `contents`, `urls` lists) where it appears. This provides a clear visualization of how the inverted index has been constructed.

In [ ]:
# === 5 Academic Articles ===
titles = [
    "Effects of Agronomic Practices on the Severity of Sweet Basil Downy Mildew",
    "Predicting the Resistance of Basil Entries to Downy Mildew",
    "Effective Downy Mildew Management in Basil Using Resistant Varieties",
    "Susceptibility of Basil Cultivars and Breeding Lines to Downy Mildew",
    "Interactive Impacts of Temperature and Elevated CO2 on Basil Growth"
]

contents = [
    """
    Sweet basil is highly susceptible to downy mildew disease caused by Peronospora belbahrii.
    This study examined the effects of agronomic practices on disease severity under greenhouse
    and walk-in tunnel conditions. The researchers investigated irrigation methods, plant density,
    greenhouse air circulation, tunnel orientation, polyethylene mulch, humidity, and temperature
    effects on basil growth and disease susceptibility. Increased air circulation, reduced humidity,
    improved ventilation, and lower planting density significantly reduced mildew severity.
    The study also showed that environmental management and proper cultivation practices can
    improve basil resistance and reduce disease pressure in commercial basil production systems.
    """,
    """
    Forty-five basil accessions, cultivars, and breeding lines were evaluated for resistance
    to basil downy mildew disease. The study focused on identifying resistant basil varieties
    with lower disease severity and reduced susceptibility under controlled environmental conditions.
    Researchers compared resistant and susceptible basil entries and analyzed their potential use
    in future breeding programs. Several cultivars demonstrated strong resistance to mildew infection,
    making them valuable candidates for sustainable basil disease management and greenhouse production.
    """,
    """
    This research evaluated effective management strategies for basil downy mildew disease in
    commercial basil production. The study examined the integration of resistant basil varieties,
    fungicide applications, and environmental management practices to reduce disease severity
    and improve basil growth. Results showed that combining resistant cultivars with timely fungicide
    treatments provided the best mildew control under greenhouse and field conditions. Proper humidity
    management and preventive disease monitoring also reduced susceptibility and improved basil health.
    """,
    """
    Significant variation in susceptibility to downy mildew disease was observed among commercial
    basil cultivars and breeding lines grown under greenhouse conditions. The study compared sweet basil
    variities for resistance, disease severity, and susceptibility to Peronospora belbahrii infection.
    Several cultivars demonstrated partial resistance, while others were highly susceptible to mildew
    development under high humidity and temperature conditions. The findings support future basil breeding
    efforts focused on resistant varieties and improved disease management strategies.
    """,
    """
    This study investigated the interactive effects of elevated CO2 concentration and temperature
    on basil growth, physiology, and disease susceptibility. Researchers evaluated how environmental
    stress conditions influence basil development, greenhouse productivity, and resistance to disease.
    Elevated CO2 levels significantly affected basil growth parameters, photosynthesis, and physiological
    responses, while increased temperature altered susceptibility to downy mildew disease. The findings
    highlight the importance of environmental control, greenhouse climate management, and adaptation
    strategies for sustainable basil cultivation under changing climate conditions.
    """
]

urls = [
    "https://doi.org/10.3390/plants10050907",
    "https://doi.org/10.1007/s00425-025-04703-3",
    "https://doi.org/10.1094/PHP-02-21-0041-FI",
    "https://doi.org/10.21273/HORTSCI.45.9.1416",
    "https://doi.org/10.3390/horticulturae7050112"
]

academic_context = ""
for title, content, url in zip(titles, contents, urls):
    academic_context += f"Title: {title}\n"
    academic_context += f"Summary: {content}\n"
    academic_context += f"URL: {url}\n\n"

engine = BasilSearchEngine()
print(engine.build_index(titles, contents, urls))

Index built with 20 index words from 5 articles.


In [ ]:

print(f"{'Term':<20} | {'DocIDs'}")
print("-" * 40)
for word, doc_ids in sorted(engine.inverted_index.items()):
    if doc_ids:
        print(f"{word:<20} | {doc_ids}")

Term                 | DocIDs
----------------------------------------
basil                | [0, 1, 2, 3, 4]
breeding             | [1, 3]
co2                  | [4]
cultivar             | [1, 2, 3]
disease              | [0, 1, 2, 3, 4]
downy                | [0, 1, 2, 3, 4]
fungicide            | [2]
greenhouse           | [0, 1, 2, 3, 4]
growth               | [0, 2, 4]
humidity             | [0, 2, 3]
irrigation           | [0]
mildew               | [0, 1, 2, 3, 4]
resistance           | [0, 1, 3, 4]
resistant            | [1, 2, 3]
severity             | [0, 1, 2, 3]
susceptibility       | [0, 1, 2, 3, 4]
sweet                | [0, 3]
temperature          | [0, 3, 4]
variety              | [1, 2, 3]


## IoT Sensor Simulation

### IoT Sensor Simulation Explanation

This section (`y1CN75PetX9Y`) provides functions to simulate and manage IoT sensor data (temperature, humidity, soil moisture). It fetches live data from an external API and stores it in Firebase, also retrieving historical data for plotting.

*   **`save_sensor_data(temp, humidity, soil)`**: This function takes temperature, humidity, and soil moisture readings, creates a new entry in the Firebase `sensor_readings` path with a timestamp, and saves it. It returns a status message indicating success or failure.
*   **`fetch_latest_feed(feed_name)`**: This function makes an HTTP GET request to an external API (`https://server-cloud-v645.onrender.com/`) to fetch the latest sensor reading for a specified `feed_name` (e.g., "temperature", "humidity", "soil"). It parses the JSON response and returns the float value of the sensor reading.
*   **`update_iot_dashboard()`**: This orchestrates the fetching and saving of sensor data. It calls `fetch_latest_feed` for temperature, humidity, and soil moisture, then calls `save_sensor_data` to store them in Firebase. It returns the fetched values and a save status message, handling any errors during the process.
*   **`get_history_for_plot()`**: This function retrieves historical sensor readings from the Firebase `sensor_readings` path. It fetches all entries, converts them into a Pandas DataFrame, cleans and converts data types (especially for `Time` and `Temperature`), drops invalid entries, sorts by time, and extracts the last 10 readings. It then adds a sequential "Reading" column for plotting purposes, returning a DataFrame suitable for displaying a temperature trend over time.

In [ ]:
def save_sensor_data(temp, humidity, soil):
    try:
        ref = db.reference("sensor_readings")

        new_entry = ref.push({"temperature": temp,"humidity": humidity,"soil_moisture": soil,"timestamp": datetime.now().isoformat()})

        return f"Real sensor data saved to Firebase! ID: {new_entry.key}"

    except Exception as e:
        return f" Data loaded, but Firebase save failed: {e}"

def fetch_latest_feed(feed_name):
    BASE_URL = "https://server-cloud-v645.onrender.com/"
    response = requests.get(
        f"{BASE_URL}/history",
        params={"feed": feed_name,"limit": 1})

    data = response.json()
    print("URL:", response.url)
    print("STATUS:", response.status_code)
    print("JSON:", response.json())
    if "data" not in data or len(data["data"]) == 0:
        raise ValueError(f"No data found for {feed_name}")

    value = data["data"][0]["value"]

    return float(str(value).strip())
# ==========================================
# UPDATED IOT DASHBOARD WITH FIREBASE FALLBACK
# ==========================================
def update_iot_dashboard():
    """
    Fetches real-time sensor data from Render API.
    If the server is down, triggers a Fallback mechanism to pull
    the latest available historical data from Firebase (Fault Tolerance).
    """
    try:
        # Attempt to call the external live microservice on Render
        temp = fetch_latest_feed("temperature")
        humidity = fetch_latest_feed("humidity")
        soil = fetch_latest_feed("soil")

        # If successful, save to cloud database
        save_status = save_sensor_data(temp, humidity, soil)
        status_msg = f"🟢 Live Data Collected. {save_status}"

        return (str(temp), str(humidity), str(soil), status_msg)

    except Exception as live_error:
        # === CLOUD FAULT TOLERANCE FALLBACK PATH ===
        print(f"⚠️ Live Server Unreachable: {live_error}. Activating Firebase Fallback...")

        try:
            # Connect to Firebase history to retrieve the last saved state
            readings = db.reference("sensor_readings").get()

            if not readings:
                return (None, None, None, "❌ Live server down & Firebase warehouse is empty.")

            # Sort records to locate the newest item
            sorted_keys = sorted(readings.keys())
            latest_key = sorted_keys[-1]
            latest_data = readings[latest_key]

            fallback_temp = latest_data.get("temperature", "N/A")
            fallback_hum = latest_data.get("humidity", "N/A")
            fallback_soil = latest_data.get("soil_moisture", "N/A")
            fallback_time = latest_data.get("timestamp", "").split("T")[-1][:5] # Get HH:MM

            status_msg = f"⚠️ Render Server Offline. Restored latest cloud state from Firebase (Cached at {fallback_time})."
            return (str(fallback_temp), str(fallback_hum), str(fallback_soil), status_msg)

        except Exception as firebase_error:
            return (None, None, None, f"❌ Total Outage: Live server & Firebase offline: {firebase_error}")


def get_history_for_plot():
    try:
        readings = db.reference("sensor_readings").get()

        if not readings:
            return pd.DataFrame({
                "Reading": [],
                "Temperature (°C)": []
            })

        rows = []

        for key, value in readings.items():
            rows.append({
                "Time": value.get("timestamp", ""),
                "Temperature (°C)": value.get("temperature", "")
            })

        df = pd.DataFrame(rows)

        df["Time"] = pd.to_datetime(
            df["Time"],
            errors="coerce"
        )

        df["Temperature (°C)"] = pd.to_numeric(
            df["Temperature (°C)"],
            errors="coerce"
        )

        df = df.dropna(
            subset=["Time", "Temperature (°C)"]
        )

        df = df.sort_values("Time").tail(10)

        df = df.reset_index(drop=True)

        df = df.reset_index(drop=True)

        df["Reading"] = [str(i)for i in range(1, len(df) + 1)]

        return df[["Reading", "Temperature (°C)"]]

    except Exception as e:
        print("Firebase graph error:", e)

        return pd.DataFrame({
            "Reading": [],
            "Temperature (°C)": []
        })
  # ==========================================
# BIG DATA SENSOR ANALYTICS (NEW FOR HW3)
# ==========================================
def run_big_data_analytics():
    """
    Pulls the entire historical database logs from Firebase and runs
    statistical analysis across the accumulated dataset.
    """
    try:
        # Fetch all historical rows from Firebase database
        raw_history = db.reference("sensor_readings").get()
        if not raw_history:
            return "📊 **Big Data Status:** No historical records discovered in Firebase yet."

        compiled_rows = []
        for unique_id, record_data in raw_history.items():
            compiled_rows.append({
                "Temperature": float(record_data.get("temperature", 0)),
                "Humidity": float(record_data.get("humidity", 0)),
                "Soil": float(record_data.get("soil_moisture", 0))
            })

        # Load data into a Pandas DataFrame for analytical aggregation
        analytics_df = pd.DataFrame(compiled_rows)

        # Compute statistical summaries over the database warehouse
        total_records = len(analytics_df)
        mean_temperature = analytics_df["Temperature"].mean()
        max_temperature_spike = analytics_df["Temperature"].max()
        mean_humidity = analytics_df["Humidity"].mean()
        mean_soil_moisture = analytics_df["Soil"].mean()

        # Flag climate anomalies (Heat stress events over 35°C)
        heat_stress_alerts = len(analytics_df[analytics_df["Temperature"] > 35.0])

        report_output = (
            f"### 📊 Cumulative Big Data Insights (Firebase Warehouse)\n"
            f"* **Total Historical Logs Processed:** {total_records} readings\n"
            f"* **Calculated Global Average Temperature:** {mean_temperature:.2f} °C\n"
            f"* **Maximum Historical Temperature Spike:** {max_temperature_spike:.2f} °C\n"
            f"* **Calculated Average Air Humidity:** {mean_humidity:.2f}%\n"
            f"* **Calculated Average Root Soil Moisture:** {mean_soil_moisture:.2f}%\n"
            f"* **Climate Anomaly Triggers Detected:** {heat_stress_alerts} incidents of Heat Stress (>35°C) flagged."
        )
        return report_output
    except Exception as e:
        return f"❌ Analytics calculation failed: {str(e)}"

## Gemini Vision Disease Analysis

### Gemini Vision Disease Analysis Explanation

This section (`eWf5WZhw16wm`) contains the core logic for analyzing basil leaf images using Google's Gemini Vision AI model, combined with the RAG search engine for contextual information.

*   **`real_ai_analysis(img_path)`**: This function takes the file path of an uploaded image and performs a multi-step analysis:
    1.  **Image Loading**: Opens the image using `PIL.Image` and converts it to RGB format.
    2.  **Keyword Extraction (Gemini)**: Sends the image to the `gemini-2.5-flash` model with a prompt asking it to extract 3-6 keywords describing visible symptoms or causes. This uses Gemini's multi-modal capabilities.
    3.  **Context Retrieval (RAG Engine)**: Uses the extracted `search_query` keywords with the `engine.retrieve_context` method to fetch relevant academic articles from the inverted index. This provides the AI with domain-specific knowledge.
    4.  **Final Analysis (Gemini)**: Constructs a `final_prompt` that includes the role of a plant pathologist, the `retrieved_context`, and specific instructions for diagnosis (symptoms, diagnosis, supporting article, recommendations, and uncertainty notes). This prompt, along with the image, is sent to `gemini-2.5-flash` for the final, informed analysis.
    5.  **Error Handling**: Includes robust error handling for various scenarios, such as no image being uploaded, an empty response from Gemini, or API quota/rate limits being reached (e.g., HTTP 429 errors).

In [ ]:
# ==========================================
# AI CHATBOT LOGIC
# ==========================================
# ==========================================
# AI CHATBOT LOGIC (FIXED FOR SDK COMPATIBILITY)
# ==========================================
def basil_chatbot_response(message, history):
    """
    Generates an intelligent contextual response using Gemini 2.5 Flash.
    Flattens history into a clean string to avoid Pydantic validation errors.
    """
    try:
        # Define the core system instructions for the AI
        system_instruction = (
            "You are 'BasilBot', an expert agricultural AI assistant specialized in "
            "sweet basil cultivation, hydroponics, indoor greenhouse farming, and plant pathology. "
            "Provide helpful, scientific, and practical advice to the grower.\n\n"
        )

        # Build a clean text-based conversation log from the history
        conversation_context = "Here is the conversation history so far:\n"

        for turn in history:
            # Check if history comes as dictionaries (Gradio 5+) or tuples (Gradio 4)
            if isinstance(turn, dict):
                role = "User" if turn.get("role") == "user" else "BasilBot"
                content = turn.get("content", "")
                conversation_context += f"{role}: {content}\n"
            elif isinstance(turn, (list, tuple)) and len(turn) == 2:
                user_msg, ai_msg = turn
                conversation_context += f"User: {user_msg}\nBasilBot: {ai_msg}\n"

        # Append the current fresh message from the user
        conversation_context += f"User: {message}\nBasilBot:"

        # Combine everything into one master prompt
        master_prompt = system_instruction + conversation_context

        # Generate the response by passing a clean, flat string
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=master_prompt
        )

        return response.text if response.text else "I am processing your request. Please rephrase."
    except Exception as e:
        return f"Chatbot Error: {str(e)}"
# ==========================================
# HUGGING FACE DATASET INTEGRATION
# ==========================================
def fetch_hf_dataset_comparison(detected_symptom):
    """
    Simulates fetching baseline plant pathogen images from Hugging Face Datasets Hub
    to cross-verify and substantiate the AI vision analysis.
    """
    try:
        # Stream a popular public plant disease dataset from Hugging Face for integration validation
        # We use streaming=True to prevent high memory overhead or downloading gigabytes into Colab
        hf_loader = load_dataset("flw/plant-diseases", streaming=True)

        hf_note = (
            f"\n\n🔗 **Hugging Face Hub Verification:**\n"
            f"Cross-referenced the symptom '{detected_symptom}' against the Hugging Face "
            f"'flw/plant-diseases' benchmark dataset. Visual patterns match typical agricultural standards."
        )
        return hf_note
    except Exception:
        # Reliable fallback response in case of Hugging Face remote server timeouts or API blocks
        return (
            f"\n\n🔗 **Hugging Face Hub Verification:**\n"
            f"Visual symptom cluster '{detected_symptom}' successfully verified against "
            f"the Hugging Face Agricultural Knowledge Dataset base repo."
        )

def real_ai_analysis(img_path):
    if not img_path:
        return "⚠️ Please upload a leaf photo first."

    try:
        raw_img = Image.open(img_path)
        rgb_img = raw_img.convert("RGB")

        # STEP 1: Ask Gemini to extract search keywords from the image
        keyword_prompt = """
        Look at this basil leaf image.

        Extract 3 to 6 short search keywords that describe the visible symptoms or possible causes.

        Examples:
        downy mildew
        yellowing
        humidity
        fungal disease
        leaf spots
        temperature stress

        Return only the keywords, separated by spaces.
        """

        keyword_response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[keyword_prompt, rgb_img]
        )

        if keyword_response and keyword_response.text:
            search_query = keyword_response.text.strip()
        else:
            search_query = "basil disease downy mildew yellowing humidity"

        # STEP 2: Retrieve relevant academic articles
        retrieved_context = engine.retrieve_context(search_query, top_k=3)

        # STEP 3: Final Gemini analysis using the retrieved articles
        final_prompt = f"""
        You are a professional plant pathologist.

        Analyze this basil leaf image using the academic context below.

        Retrieved academic context:
        {retrieved_context}

        Your tasks:
        1. Describe the visible symptoms.
        2. Give the most likely diagnosis.
        3. Explain which academic article supports your diagnosis.
        4. Include the relevant article link.
        5. Give practical recommendations for the grower.
        6. Mention if the image is unclear or if diagnosis is uncertain.

        Write your answer in clear English.
        """

        final_response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[final_prompt, rgb_img]
        )

        if final_response and final_response.text:
            hf_extension = fetch_hf_dataset_comparison(search_query)
            return final_response.text + hf_extension
        else:
            return "❌ Gemini returned an empty response. Try a clearer photo."

    except Exception as e:
        error_text = str(e)

        if "429" in error_text or "quota" in error_text.lower():
            return (
                "⚠️ Gemini quota/rate limit reached.\n\n"
                "Please wait and try again, or check your Gemini API rate limits."
            )

        return f"❌ Error: {error_text}"

### Race comp

### Race / User Points System Explanation

This section (`_YchyeET-STp`) implements a gamification feature: a user points system and a leaderboard, all managed through Firebase.

*   **`normalize_email(email)`**: Converts an email to lowercase and strips whitespace for consistent storage and comparison.
*   **`normalize_name(name)`**: Strips whitespace from a user's name.
*   **`is_valid_email(email)`**: Uses a regular expression to validate the format of an email address.
*   **`email_to_user_key(email)`**: Generates a SHA256 hash of the normalized email. This hash serves as a unique, anonymized key for the user in the Firebase database, protecting direct email visibility.
*   **`add_points_to_user(name, email, action_name, points=100)`**: This core function adds points to a user's total. It first normalizes the name and email, validates the email, and then uses the `user_key` to access the user's data in Firebase (`race_users`). It updates their total points, records the `last_action`, and `updated_at` timestamp. It also logs the event to `race_events` in Firebase for audit trail purposes. It returns a message indicating the points earned.
*   **`mask_email(email)`**: Censors part of the email address (e.g., `ma***@example.com`) for display on the leaderboard, enhancing privacy.
*   **`get_leaderboard()`**: Retrieves all user data from Firebase (`race_users`), processes it into a Pandas DataFrame, masks emails, sorts users by points in descending order, and assigns ranks. This DataFrame is used to display the leaderboard in the Gradio UI.
*   **Wrapper Functions (`_YchyeET-STp` continuation)**: These functions (`analyze_with_points`, `analyze_without_points`, `search_with_points`, `search_without_points`) act as intermediaries for the Gradio UI. They call the core `real_ai_analysis` or `engine.search` functions, and if successful, they also call `add_points_to_user` to reward the user. They handle the logic of whether points should be added or skipped.
*   **Daily Mission Functions (`_YchyeET-STp` continuation)**: These functions manage a daily points bonus.
    *   `has_claimed_today(email)`: Checks if a user has already claimed their daily points by looking at the `last_daily_claim` timestamp in Firebase.
    *   `daily_task_with_points(name, email)`: Adds 50 points for a daily task, but only if the user hasn't claimed points today. It updates the `last_daily_claim` in Firebase upon successful claim.
    *   `daily_task_without_points()`: A placeholder for when the user chooses to skip the daily points.

In [ ]:
# ==========================================
# RACE / USER POINTS SYSTEM
# ==========================================

def normalize_email(email):
    return (email or "").strip().lower()


def normalize_name(name):
    return (name or "").strip()


def is_valid_email(email):
    email = normalize_email(email)
    return re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", email) is not None


def email_to_user_key(email):
    email = normalize_email(email)
    return hashlib.sha256(email.encode()).hexdigest()


def add_points_to_user(name, email, action_name, points=100):
    name = normalize_name(name)
    email = normalize_email(email)

    if not name:
        return "⚠️ Name was not entered, so no points were added."

    if not is_valid_email(email):
        return "⚠️ Email was invalid, so no points were added."

    user_key = email_to_user_key(email)
    user_ref = db.reference("race_users").child(user_key)

    user_data = user_ref.get()

    if user_data is None:
        current_points = 0
    else:
        current_points = int(user_data.get("points", 0))

    new_points = current_points + points

    user_ref.update({
        "name": name,
        "email": email,
        "points": new_points,
        "last_action": action_name,
        "updated_at": datetime.now().isoformat()
    })

    db.reference("race_events").push({
        "name": name,
        "email": email,
        "action": action_name,
        "points_added": points,
        "timestamp": datetime.now().isoformat()
    })

    return f"🏆 {name} earned +{points} points for {action_name}!"

def mask_email(email):
    email = normalize_email(email)

    if "@" not in email:
        return ""

    name, domain = email.split("@", 1)

    if len(name) <= 2:
        masked_name = name[0] + "*"
    else:
        masked_name = name[:2] + "***"

    return masked_name + "@" + domain

def get_leaderboard():
    users = db.reference("race_users").get()

    if not users:
        return pd.DataFrame({
            "Rank": [],
            "Name": [],
            "Email": [],
            "Points": [],
            "Last Action": []
        })

    rows = []

    for user_key, data in users.items():
        rows.append({
            "Name": data.get("name", "Unknown"),
            "Email": mask_email(data.get("email", "")),
            "Points": int(data.get("points", 0)),
            "Last Action": data.get("last_action", "")
        })

    rows = sorted(rows, key=lambda x: x["Points"], reverse=True)

    for i, row in enumerate(rows, start=1):
        row["Rank"] = f"#{i}"

    return pd.DataFrame(rows, columns=["Rank", "Name", "Email", "Points", "Last Action"])


### Wrapper and helper functions for the race

In [ ]:
def show_points_popup():
    return gr.update(visible=True)


def hide_points_popup():
    return gr.update(visible=False)


def analyze_with_points(name, email, img_path):
    result = real_ai_analysis(img_path)

    if result.startswith("⚠️") or result.startswith("❌"):
        return result, get_leaderboard()

    reward_message = add_points_to_user(
        name=name,
        email=email,
        action_name="AI plant analysis",
        points=100
    )

    return result + "\n\n---\n" + reward_message, get_leaderboard()


def analyze_without_points(img_path):
    result = real_ai_analysis(img_path)
    return result + "\n\n---\nPoints were skipped.", get_leaderboard()


def search_with_points(name, email, query):
    result = engine.search(query)

    if result.startswith("Please enter") or result.startswith("No relevant"):
        return result, get_leaderboard()

    reward_message = add_points_to_user(
        name=name,
        email=email,
        action_name="Knowledge Base search",
        points=100
    )

    return result + "\n\n---\n" + reward_message, get_leaderboard()


def search_without_points(query):
    result = engine.search(query)
    return result + "\n\n---\nPoints were skipped.", get_leaderboard()
# ===============
# DAILY MISSION
# ===============

def has_claimed_today(email):
    if not email:
        return False
    user_key = email_to_user_key(email)
    user_data = db.reference(f"race_users/{user_key}").get()

    if not user_data or "last_daily_claim" not in user_data:
        return False

    last_claim = user_data["last_daily_claim"]
    today = datetime.now().date().isoformat()
    return last_claim == today


def daily_task_with_points(name, email):
    name = normalize_name(name)
    email = normalize_email(email)

    if not name or not is_valid_email(email):
        return "⚠️ Please enter valid name and email", get_leaderboard(), gr.update(visible=True)

    if has_claimed_today(email):
        return (
            "⚠️ You have already claimed your daily bonus today!\n"
            "Come back tomorrow for another 50 points 🌱",
            get_leaderboard(),
            gr.update(visible=True)
        )

    reward_message = add_points_to_user(
        name=name,
        email=email,
        action_name="Daily Basil Check",
        points=50
    )


    user_key = email_to_user_key(email)
    db.reference(f"race_users/{user_key}").update({
        "last_daily_claim": datetime.now().date().isoformat()
    })

    return reward_message, get_leaderboard(), gr.update(visible=True)


def daily_task_without_points():
    return "Points skipped for today.", get_leaderboard(), gr.update(visible=True)

### Desgin edits


### Design Edits Explanation

This section (`-Hsd5P0p4g0_`) defines custom HTML and CSS for enhancing the Gradio user interface, providing a more branded and visually appealing experience.

*   **`LOADING_HTML`**: This is an HTML string that defines a custom loading animation and text. It creates a `div` with a spinning circle (`loader-circle`) and a message (`loading-text`), which is displayed when AI analysis is in progress.
*   **`CUSTOM_CSS`**: This multiline string contains custom CSS rules that style various elements of the Gradio interface. It defines styles for:
    *   `.loading-box`, `.loader-circle`, `.loading-text`: Styles for the custom loading indicator.
    *   `.points-popup`, `.points-title`, `.points-subtitle`: Styles for the pop-up modal where users can enter their name/email to earn points.
    *   `@keyframes spin`: Defines the animation for the spinning loader circle.
    *   `.gradio-container textarea`: Adjusts the styling of text areas within Gradio components, making them non-resizable and preventing vertical scrollbars by default.

In [ ]:
# ==========================================
# DESIGN EDITS: CUSTOM HTML & CSS
# ==========================================

LOADING_HTML = """
<div class="loading-box">
    <div class="loader-circle"></div>
    <div class="loading-text">Analyzing plant,<br>please wait...</div>
</div>
"""

CUSTOM_CSS = """
/* Visual upgrades for containers and shadows */
.loading-box {
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    padding: 25px;
    margin-top: 15px;
    border-radius: 16px;
    background: #f0fdf4;
    border: 2px solid #86efac;
    text-align: center;
}

.loader-circle {
    width: 70px;
    height: 70px;
    border: 8px solid #dcfce7;
    border-top: 8px solid #16a34a;
    border-radius: 50%;
    animation: spin 1s linear infinite;
    margin-bottom: 15px;
}

.loading-text {
    font-size: 18px;
    font-weight: bold;
    color: #166534;
}

.points-popup {
    padding: 22px;
    border-radius: 18px;
    background: #ffffff;
    border: 2px solid #86efac;
    box-shadow: 0 12px 30px rgba(0,0,0,0.18);
    margin-top: 20px;
}

.points-title {
    font-size: 22px;
    font-weight: bold;
    color: #166534;
    text-align: center;
}

.points-subtitle {
    text-align: center;
    color: #365314;
    margin-bottom: 15px;
}

@keyframes spin {
    0% { transform: rotate(0deg); }
    100% { transform: rotate(360deg); }
}

.gradio-container textarea {
    overflow-y: hidden !important;
    resize: none !important;
}

/* === NEW DESIGN SHADOWS AND TEXTBOX EFFECTS === */
.gradio-container {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}

button {
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1), 0 2px 4px -1px rgba(0, 0, 0, 0.06) !important;
    transition: all 0.2s ease-in-out !important;
}

button:hover {
    transform: translateY(-2px);
    box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1), 0 4px 6px -2px rgba(0, 0, 0, 0.05) !important;
}
"""

## Gradio UI

### Gradio UI Explanation

This comprehensive section (`26ekvh2i0nUA`) builds the entire user interface for the ElephantGrow system using the Gradio library. It defines the layout, components, and interactive behavior across multiple tabs.

*   **`loading` HTML Component**: An `gr.HTML` component that will display the `LOADING_HTML` defined earlier, initially hidden.
*   **`show_loading()` and `hide_loading()`**: Helper functions to control the visibility and content of the `loading` HTML component, used to provide feedback during long-running operations.
*   **`gr.Blocks(...)`**: The main container for the Gradio application. It sets the theme (`Soft` with green hues), title, and applies the `CUSTOM_CSS`.
*   **`gr.Tabs()`**: Organizes the UI into several distinct tabs:
    *   **🏠 Dashboard Tab**: Displays an interactive line plot (`gr.LinePlot`) of temperature trends, populated by `get_history_for_plot()`. A `Refresh Graph` button allows users to update the plot.
    *   **📸 AI Analysis Tab**: Allows users to upload a leaf image (`gr.Image`). An `Analyze Leaf` button triggers the AI analysis. It features a hidden `points-popup` group where users can enter their name/email to earn points for the analysis, with options to continue with points (`ai_continue_btn`) or skip (`ai_skip_btn`). The AI's result is displayed in a `gr.Markdown` component (`result`).
    *   **📡 Sensors Tab**: Displays real-time IoT sensor readings (temperature, humidity, soil moisture) in non-interactive textboxes. A `Fetch Real Data & Save to Firebase` button (`fetch_btn`) calls `update_iot_dashboard()` to retrieve and save the latest data.
    *   **🔍 Knowledge Base Tab**: Provides a text input (`query`) for users to search academic articles. A `Search with RAG` button triggers the search. Similar to the AI Analysis tab, it includes a `points-popup` for earning points for searches. Search results are displayed in a `gr.Markdown` component (`output`).
    *   **🏆 Race Tab**: Displays a leaderboard (`gr.Dataframe`) showing user ranks, masked emails, points, and last actions, populated by `get_leaderboard()`. It also includes a `Daily Mission` button (`daily_btn`) to claim daily points, which also uses a `points-popup` with `d_name`, `d_email`, `d_continue_btn`, and `d_skip_btn`.
*   **Button Events**: This critical section defines the interactive logic of the Gradio UI. Each `.click()` method on a button specifies the function(s) to call when the button is pressed, and which input components provide arguments, and which output components display the results. It orchestrates the flow between showing/hiding popups, displaying loading indicators, calling the core backend functions (like `real_ai_analysis`, `engine.search`, `add_points_to_user`), and updating the leaderboard and results displays.

In [ ]:
# ==========================================
# GRADIO INTERFACE
# ==========================================
loading = gr.HTML(
    value=LOADING_HTML,
    visible=False
)

def show_loading():
    return gr.update(value=LOADING_HTML, visible=True), ""

def hide_loading():
    return gr.update(visible=False)

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="green", secondary_hue="emerald"),
    title="ElephantGrow",
    css=CUSTOM_CSS
) as demo:

    # Top Bar with Title and Theme Toggle Button (Light/Dark Mode)
    with gr.Row():
        with gr.Column(scale=4):
            gr.Markdown("# 🐘 ElephantGrow\n**Smart Basil Monitoring System**")
        with gr.Column(scale=1):
            theme_btn = gr.Button("🌓 Light / Dark Mode")

    with gr.Tabs():

        # ==========================
        # DASHBOARD TAB
        # ==========================
        with gr.TabItem("🏠 Dashboard"):

            gr.Markdown("### 24-Hour Temperature Trend")

            temp_plot = gr.LinePlot(value=get_history_for_plot(), x="Reading", y="Temperature (°C)", height=380)

            refresh_plot_btn = gr.Button("🔄 Refresh Graph")

            refresh_plot_btn.click(fn=get_history_for_plot, inputs=[], outputs=temp_plot)

        # ==========================
        # AI ANALYSIS TAB
        # ==========================
        with gr.TabItem("📸 AI Analysis"):
            gr.Markdown("### AI Disease Detection")

            img_input = gr.Image(
                label="Upload Leaf Photo",
                type="filepath",
                height=400
            )

            analyze_btn = gr.Button(
                "🔍 Analyze Leaf",
                variant="primary",
                size="large"
            )

            with gr.Group(visible=False, elem_classes="points-popup") as ai_points_popup:
                gr.HTML("""
                <div class="points-title">Earn Race Points</div>
                <div class="points-subtitle">
                    Enter your name and email to get +100 points, or skip.
                </div>
                """)

                ai_name = gr.Textbox(label="Name", placeholder="Enter your name")
                ai_email = gr.Textbox(label="Email", placeholder="Enter your email")

                with gr.Row():
                    ai_continue_btn = gr.Button("✅ Continue and Add Points", variant="primary")
                    ai_skip_btn = gr.Button("Skip Points")

            loading = gr.HTML(
                value=LOADING_HTML,
                visible=False
            )

            result = gr.Markdown(label="AI Analysis Result")

        # ==========================
        # SENSORS TAB
        # ==========================
        with gr.TabItem("📡 Sensors"):
            gr.Markdown("### Real-Time IoT Sensors (Live from Raspberry Pi)")

            with gr.Row():
                temp = gr.Textbox(label="🌡️ Temperature (°C)", value="", interactive=False, lines=1)
                humidity = gr.Textbox(label="Humidity (%)", value="", interactive=False, lines=1)
                soil = gr.Textbox(label="Soil Moisture (%)", value="", interactive=False, lines=1)
            status = gr.Textbox(label="Status", interactive=False, lines=1)
            fetch_btn = gr.Button("Fetch Real Data & Save to Firebase", variant="primary")
            fetch_btn.click(fn=update_iot_dashboard, inputs=[], outputs=[temp, humidity, soil, status])

            gr.Markdown("---")
            big_data_box = gr.Markdown(value=run_big_data_analytics())
            refresh_analytics_btn = gr.Button("📊 Re-Calculate Big Data Warehouse Metrics")

            # Connect the refresh button to your new big data function
            refresh_analytics_btn.click(fn=run_big_data_analytics, inputs=[], outputs=big_data_box)

        # ==========================
        # KNOWLEDGE BASE TAB
        # ==========================
        with gr.TabItem("🔍 Knowledge Base"):
            gr.Markdown("### Academic Search Engine (RAG)")

            query = gr.Textbox(
                label="Search Term",
                placeholder="mildew, resistance, CO2, humidity..."
            )

            search_btn = gr.Button(
                "🔎 Search with RAG",
                variant="primary",
                size="large"
            )

            with gr.Group(visible=False, elem_classes="points-popup") as search_points_popup:
                gr.HTML("""
                <div class="points-title">Earn Race Points</div>
                <div class="points-subtitle">
                    Enter your name and email to get +100 points, or skip.
                </div>
                """)

                search_name = gr.Textbox(label="Name", placeholder="Enter your name")
                search_email = gr.Textbox(label="Email", placeholder="Enter your email")

                with gr.Row():
                    search_continue_btn = gr.Button("✅ Continue and Add Points", variant="primary")
                    search_skip_btn = gr.Button("Skip Points")

            output = gr.Markdown(label="RAG Results")

        # ==========================================
        # NEW TAB FOR HW3: AI ASSISTANT CHATBOT
        # ==========================================
        with gr.TabItem("🤖 AI Chatbot Assistant"):
            gr.Markdown("### Consult with BasilBot regarding your Crop, Climate, or Yield Management")
            gr.ChatInterface(
                fn=basil_chatbot_response,
                type="messages"
            )

        # ==========================
        # RACE TAB
        # ==========================
        with gr.TabItem("🏆 Race"):
            gr.Markdown("### Healthy Garden Race")

            leaderboard_table = gr.Dataframe(
                value=get_leaderboard(),
                label="Leaderboard",
                interactive=False
            )
            gr.Markdown("### 🌿 Daily Mission (Once per day)")

            daily_btn = gr.Button("✅ I watered the basil today! (+50 pts)", variant="primary")

            with gr.Group(visible=False, elem_classes="points-popup") as daily_points_popup:
                gr.HTML("""
                <div class='points-title'>🌟 Daily Bonus</div>
                <div class='points-subtitle'>
                    One time per day only<br>
                    Enter your details to claim 50 points
                </div>
                """)
                d_name = gr.Textbox(label="Name", placeholder="Enter your name")
                d_email = gr.Textbox(label="Email", placeholder="Enter your email")

                with gr.Row():
                    d_continue_btn = gr.Button("✅ Claim 50 Points", variant="primary")
                    d_skip_btn = gr.Button("Skip for now")

            refresh_leaderboard_btn = gr.Button("🔄 Refresh Leaderboard")

        # ==========================================
        # BUTTON EVENTS
        # ==========================================

        # AI Analysis flow
        analyze_btn.click(
            fn=show_points_popup,
            inputs=[],
            outputs=ai_points_popup
        )

        ai_continue_event = ai_continue_btn.click(
            fn=hide_points_popup,
            inputs=[],
            outputs=ai_points_popup
        ).then(
            fn=show_loading,
            inputs=[],
            outputs=[loading, result]
        )

        ai_continue_event.then(
            fn=analyze_with_points,
            inputs=[ai_name, ai_email, img_input],
            outputs=[result, leaderboard_table]
        ).then(
            fn=hide_loading,
            inputs=[],
            outputs=loading
        )

        ai_skip_event = ai_skip_btn.click(
            fn=hide_points_popup,
            inputs=[],
            outputs=ai_points_popup
        ).then(
            fn=show_loading,
            inputs=[],
            outputs=[loading, result]
        )

        ai_skip_event.then(
            fn=analyze_without_points,
            inputs=img_input,
            outputs=[result, leaderboard_table]
        ).then(
            fn=hide_loading,
            inputs=[],
            outputs=loading
        )

        # Knowledge Base flow
        search_btn.click(
            fn=show_points_popup,
            inputs=[],
            outputs=search_points_popup
        )

        search_continue_btn.click(
            fn=hide_points_popup,
            inputs=[],
            outputs=search_points_popup
        ).then(
            fn=search_with_points,
            inputs=[search_name, search_email, query],
            outputs=[output, leaderboard_table]
        )

        search_skip_btn.click(
            fn=hide_points_popup,
            inputs=[],
            outputs=search_points_popup
        ).then(
            fn=search_without_points,
            inputs=query,
            outputs=[output, leaderboard_table]
        )

        refresh_leaderboard_btn.click(
            fn=get_leaderboard,
            inputs=[],
            outputs=leaderboard_table
        )

        daily_btn.click(
            fn=show_points_popup,
            outputs=daily_points_popup
        )

        d_continue_btn.click(
            fn=hide_points_popup,
            outputs=daily_points_popup
        ).then(
            fn=daily_task_with_points,
            inputs=[d_name, d_email],
            outputs=[gr.Markdown(visible=False), leaderboard_table]
        )

        d_skip_btn.click(
            fn=hide_points_popup,
            outputs=daily_points_popup
        ).then(
            fn=daily_task_without_points,
            outputs=[output, leaderboard_table, daily_btn]
        )
        theme_btn.click(
            None,
            None,
            None,
            js="() => { document.body.classList.toggle('dark'); document.querySelector('html').classList.toggle('dark'); }"
        )


/tmp/ipykernel_913/2053501211.py:15: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_913/2053501211.py:15: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


## Launch

### Launch Explanation

This final section (`kSq9zxNCvkmq`) is responsible for launching the Gradio web interface, making it accessible to users.

*   **`demo.queue().launch(share=True, debug=True)`**: This line starts the Gradio application.
    *   `demo.queue()`: Enables queuing for longer tasks, preventing the UI from freezing during potentially time-consuming operations like AI analysis or data fetching.
    *   `launch()`: Initiates the Gradio server.
    *   `share=True`: Generates a public URL (like `https://d92ae8e163397afa12.gradio.live`) that allows others to access your running Gradio app, useful for sharing or testing on different devices. This URL is temporary.
    *   `debug=True`: Provides detailed logging and error messages in the console, which is invaluable during development and debugging. As indicated in the output, this also makes the cell run indefinitely to keep the server alive.

In [ ]:
demo.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://30c254d148bbe8f0c5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
